# Train pipeline

## Setup envoironment

### Colab

In [ ]:
!git clone https://github.com/trxxnk/text-image-alignment.git

In [ ]:
import os
os.chdir("/content/text-image-alignment/")
print(f"Working directory: {os.getcwd()}")

In [ ]:
!git checkout dev

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!chmod +x src/scripts/setup_colab.sh
!src/scripts/setup_colab.sh

### Local

In [1]:
import os, sys
os.chdir(os.path.dirname(sys.prefix))
print(f"Working directory: {os.getcwd()}")

Working directory: /home/trxxnk/mycode/diplom


## Import libs

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision
from torchvision.transforms import v2

import os
import json
import mlflow
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

In [4]:
from src.tps_dewarp.training import (
    Trainer,
    load_train_config,
    build_tps_dataloaders,
    build_model,
)


## Setup torch, dugshub, mlflow

In [4]:
SEED = 42
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
import dagshub
dagshub.init(repo_owner='trxxnk', repo_name='text-image-alignment', mlflow=True)

Accessing as trxxnk

Initialized MLflow to track repo "trxxnk/text-image-alignment"

Repository trxxnk/text-image-alignment initialized!

## Dataset

### Load Dataset

In [5]:
CONFIG_PATH = "configs/train_default.yaml"
cfg = load_train_config(CONFIG_PATH)


In [6]:
train_loader, val_loader, test_loader = build_tps_dataloaders(cfg, device)
len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset)


33428

In [7]:
# Проверка одного элемента train
img, delta, difficulty = train_loader.dataset[0]

print(img.shape)
print(delta.shape)
print(difficulty)


torch.Size([1, 256, 256])
torch.Size([81, 2])
hard


In [8]:
from collections import Counter

subset = train_loader.dataset
diffs = [subset.dataset.samples[i]["difficulty"] for i in subset.indices]
counter = Counter(diffs)
total = counter.total()
probas = [val / total for val in counter.values()]
print(f"""
  {counter}
  {total=}
  probas={[round(val, 2) for val in probas]}
"""
)



  Counter({'easy': 13529, 'medium': 6763, 'hard': 6583, 'identity': 6553})
  total=33428
  probas=[0.2, 0.4, 0.2, 0.2]



### Train / Val / Test split

In [12]:
# Train/val/test split и DataLoader — внутри build_tps_dataloaders(cfg, device)


(26742, 3342, 3344)

### Batching (DataLoaders)

In [ ]:
# см. ячейку выше


In [14]:
x, y, d = next(iter(train_loader))

print(x.shape)  # (B, 1, H, H)
print(y.shape)
print(len(d))


torch.Size([128, 1, 256, 256])
torch.Size([128, 81, 2])
128


## Model

### Load Model

In [15]:
model = build_model(cfg).to(device)


In [16]:
x = train_loader.dataset[1][0]
x = x.unsqueeze(0).to(device)
out = model(x)
out.shape


torch.Size([1, 162])

In [17]:
train_loader.dataset[1][1].shape


torch.Size([81, 2])

### MLflow run, затем Trainer


In [ ]:
mlflow.set_experiment("TPS_Dewarp");
mlflow.start_run(run_name="02_colab")

<ActiveRun: >

In [32]:
# Параметры логируются в MLflow из Trainer (log_params); при необходимости добавьте теги:
# mlflow.set_tags({"notebook": "03_train_model"})


In [ ]:
# grad_clip_norm и др. — в configs/train_default.yaml


In [ ]:
# Loss, optimizer, scheduler и цикл — внутри Trainer (см. configs/train_default.yaml)
trainer = Trainer(cfg, model, train_loader, val_loader, test_loader, device)


## Train Loop

In [ ]:
# Resume: trainer.fit(resume_from="models/checkpoints/last.pt")
trainer.fit(resume_from=None)


In [ ]:
# Завершить эксперимент по run_id
import mlflow
run_id = "***"
client = mlflow.tracking.MlflowClient()
client.set_terminated(run_id)